In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# South Africa Labour Market Analysis — QLFS 2026 Q1

## Project overview

This notebook analyses South Africa's **Quarterly Labour Force Survey (QLFS) 2026 Q1** unit-record data, with a particular focus on **new entrants to unemployment**.

The analysis uses:

- **Python**
- **pandas** for data manipulation and analysis
- **NumPy** for data cleaning
- **Matplotlib** for visualisation

### Main research questions

1. How does the new-entrant share vary across South African provinces?
2. How does the new-entrant share differ between males and females across provinces?
3. How does the new-entrant share vary across education statuses?

### Analytical workflow

**Understand → Validate → Clean → Explore → Analyse → Visualise → Report**

> **Important:** The raw QLFS microdata is not included in this public repository. The notebook is provided to demonstrate the analysis process. The dataset should be obtained from the official Stats SA source before rerunning the notebook.


## Load QLFS2026 Q1 Data

In [2]:
# Load the QLFS 2026 Q1 data

from pathlib import Path

# The public repository does not include the raw QLFS microdata.
# Download the official unit-record file from Stats SA and update this path if needed.
DATA_PATH = Path("../data/QLFS202601.csv")

df = pd.read_csv(DATA_PATH)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '../data/QLFS202601.csv'

In [ ]:
# get a feel for the data
print("Shape/Size: ",df.shape)
print("columns:",df.columns)
print("data types: ",df.dtypes)
print(df.isnull().sum())
print(df.head())

In [ ]:
df.info()

In [ ]:
df.nunique().sort_values()

In [ ]:
df.describe()

In [ ]:
df["Underempl"].head(20)

In [ ]:
df["Q13GENDER"].value_counts()

In [ ]:
df["Q14AGE"].value_counts().sort_index()

In [ ]:
df["Q15POPULATION"].value_counts().sort_index()

In [ ]:
df["Education_status"].value_counts().sort_index()

In [ ]:
df["Hrswrk"].describe()

In [ ]:
df["Q14AGE"].value_counts().sort_index().head(20)

In [ ]:
df["Hrswrk"].value_counts().sort_values(ascending=False).head(20)

In [ ]:
df["Weight"].describe()

In [ ]:
df["Hrswrk"].value_counts().head(10)

In [ ]:
df["Hrswrk"].sort_values().tail(10)

In [ ]:
df["Hrswrk"].describe()

In [ ]:
df[df["Hrswrk"].duplicated()]

In [ ]:
df[df["Hrswrk"] == "1.79769313486232e+308"]["Q13GENDER"].value_counts()

In [ ]:
df[df["Hrswrk"] == "1.79769313486232e+308"]["Q14AGE"].describe()

In [ ]:
df[df["Hrswrk"] == "1.79769313486232e+308"]["Education_status"].value_counts()

In [ ]:
df[df["Hrswrk"] == "1.79769313486232e+308"]["Q14AGE"].describe()

In [ ]:
df["Hrswrk"] = df["Hrswrk"].replace("1.79769313486232e+308", np.nan)

In [ ]:
df["Hrswrk"].describe()

In [ ]:
df["Hrswrk"].head()

In [ ]:
df["Q14AGE"].value_counts().sort_values(ascending=False)

In [ ]:
df[df["Q14AGE"] >= 100][
    ["Q14AGE", "Q13GENDER", "Q15POPULATION", "Education_status", "Hrswrk"]
]

In [ ]:
df["Q12NIGHTS"].value_counts().sort_index()

In [ ]:
df["Q12NIGHTS"].unique()

In [ ]:
df["Q12NIGHTS"].value_counts(dropna=False)

In [ ]:
df["Q15POPULATION"].value_counts(dropna=False).sort_index()

In [ ]:
population_counts = df["Q15POPULATION"].value_counts()
population_counts

In [ ]:
population_percent = population_counts / len(df) * 100
population_percent

In [ ]:
pd.crosstab(df["Q15POPULATION"], df["Q13GENDER"])

In [ ]:
pd.crosstab(
    df["Q15POPULATION"],
    df["Q13GENDER"],
    normalize="index"
) * 100

In [ ]:
df["Hrswrk"] = pd.to_numeric(df["Hrswrk"],errors="coerce")

In [ ]:
gender_hours = df.groupby("Q13GENDER")["Hrswrk"].mean()
gender_hours

In [ ]:
df.groupby("Q13GENDER")["Hrswrk"].describe()

In [ ]:
df.boxplot(column="Hrswrk", by="Q13GENDER")
plt.title("Hours Worked by Gender")
plt.suptitle("")
plt.xlabel("Gender")
plt.ylabel("Hours Worked per Week")
plt.show()

In [ ]:
df["Hrswrk"].nlargest(20)

In [ ]:
df["Hrswrk"].quantile([0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
df.groupby("Education_status")["Hrswrk"].agg(
    ["count", "mean", "median"]
)

In [ ]:
df.groupby("Education_status")["Hrswrk"].agg(
    mean="mean",
    median="median",
    std="std"
)


In [ ]:
df.boxplot(column="Hrswrk", by="Education_status")

plt.title("Hours Worked by Education Status")
plt.suptitle("")
plt.xlabel("Education Status")
plt.ylabel("Hours Worked per Week")
plt.show()

In [ ]:
df.groupby("Education_status")["Hrswrk"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

In [ ]:
df["Education_status"].value_counts().sort_index()

In [ ]:
pd.crosstab(
    df["Education_status"],
    df["Hrswrk"].notna(),
    normalize="index"
) * 100

In [ ]:
df.groupby("Education_status")["Hrswrk"].agg(
    ["count", "mean", "median"]
)

In [ ]:
working_hours_pct = (
    df.groupby("Education_status")["Hrswrk"]
      .apply(lambda x: x.notna().mean() * 100)
)

working_hours_pct.plot(kind="bar")

plt.title("Percentage with Recorded Working Hours by Education Status")
plt.xlabel("Education Status")
plt.ylabel("Percentage")
plt.xticks(rotation=0)
plt.show()

In [ ]:
working_hours_pct

In [ ]:
[column for column in df.columns if "emp" in column.lower()]

In [ ]:
[column for column in df.columns if "work" in column.lower()]

## Unemployment-status analysis

In [ ]:
df["Unempl_Status"].value_counts().sort_index()

In [ ]:
pd.crosstab(
    df["Education_status"],
    df["Unempl_Status"],
    normalize="index"
) * 100

In [ ]:
df["Unempl_Status"].replace(
    "1.79769313486232e+308",
    np.nan
).isna().sum()

In [ ]:
df["Unempl_Status"].replace(
    "1.79769313486232e+308",
    np.nan
).value_counts().sort_index()

In [ ]:
df["Unempl_Status"] = df["Unempl_Status"].replace(
    "1.79769313486232e+308",
    np.nan
)

In [ ]:
df["Unempl_Status"].value_counts(dropna=False).sort_index()

In [ ]:
pd.crosstab(
    df["Education_status"],
    df["Unempl_Status"],
    normalize="index"
) * 100

In [ ]:
new_entrant_pct = (
    pd.crosstab(
        df["Education_status"],
        df["Unempl_Status"],
        normalize="index"
    ) * 100
)

new_entrant_pct["3"]

In [ ]:
new_entrant_pct.plot(kind="line", marker="o")
plt.title("New Entrants by Education Status")
plt.xlabel("Education Status")
plt.ylabel("Percentage of New Entrants")
plt.grid()
plt.show()

In [ ]:
new_entrant = df["Unempl_Status"] == 3

new_entrant_by_education = (
    df.groupby("Education_status")["Unempl_Status"]
      .apply(lambda x: (x == "3").mean() * 100)
)

new_entrant_by_education

In [ ]:
highest = new_entrant_by_education.max()
lowest = new_entrant_by_education.min()

difference = highest - lowest

difference

In [ ]:
valid_unemployment = df[df["Unempl_Status"].notna()]

valid_unemployment["Education_status"].value_counts().sort_index()

In [ ]:
df["Weight"].describe()

In [ ]:
weighted_new_entrant = (
    df.groupby("Education_status")
      .apply(lambda x: (x["Weight"] * (x["Unempl_Status"] == "3")).sum()
             / x.loc[x["Unempl_Status"].notna(), "Weight"].sum() * 100)
)

weighted_new_entrant

In [ ]:
comparison = pd.DataFrame({
    "Unweighted": new_entrant_by_education,
    "Weighted": weighted_new_entrant
})

comparison["Difference"] = (
    comparison["Weighted"] - comparison["Unweighted"]
)

comparison

In [ ]:
weighted_new_entrant.plot(kind="bar")

plt.title("Weighted New Entrants by Education Status")
plt.xlabel("Education Status")
plt.ylabel("Weighted Percentage")
plt.show()

In [ ]:
pd.crosstab(
    df["Education_status"],
    df["Unempl_Status"],
    normalize="index"
) * 100

In [ ]:
unemployment_by_education = (
    pd.crosstab(
        df["Education_status"],
        df["Unempl_Status"],
        normalize="index"
    ) * 100
)

unemployment_by_education

In [ ]:
unemployment_by_education.plot(kind="bar")

plt.title("Unemployment Status by Education Level")
plt.xlabel("Education Status")
plt.ylabel("Percentage")
plt.legend(title="Unemployment Status")
plt.show()

In [ ]:
valid_unemployment = df[df["Unempl_Status"].notna()]

In [ ]:
weighted_unemployment = (
    valid_unemployment.groupby("Unempl_Status")["Weight"].sum()
    / valid_unemployment["Weight"].sum()
    * 100
)

weighted_unemployment

In [ ]:
new_entrants = df[df["Unempl_Status"] == "3"]

new_entrants["Q14AGE"].describe()

In [ ]:
new_entrants["Q14AGE"].plot(kind="hist", bins=15,rwidth=10)

plt.title("Age Distribution of New Entrants")
plt.xlabel("Age")
plt.ylabel("Number of New Entrants")
plt.show()

In [ ]:
bins = [15, 20, 25, 30, 35, 40, 50, 65, 116]
labels = ["15-19", "20-24", "25-29", "30-34",
          "35-39", "40-49", "50-64", "65+"]

new_entrants["Age_Group"] = pd.cut(
    new_entrants["Q14AGE"],
    bins=bins,
    labels=labels,
    right=False
)

new_entrants["Age_Group"].value_counts().sort_index()

In [ ]:
age_group_pct = (
    new_entrants["Age_Group"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

age_group_pct

In [ ]:
age_group_pct.plot(kind="bar")

plt.title("Percentage of New Entrants by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
weighted_age = (
    new_entrants.groupby("Age_Group")["Weight"].sum()
    / new_entrants["Weight"].sum()
    * 100
)

weighted_age

In [ ]:
age_20_24 = new_entrants[
    (new_entrants["Q14AGE"] >= 20) &
    (new_entrants["Q14AGE"] < 25)
]

age_20_24["Education_status"].value_counts().sort_index()

In [ ]:
education_20_24_pct = (
    age_20_24["Education_status"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

education_20_24_pct

In [ ]:
pd.crosstab(
    new_entrants["Age_Group"],
    new_entrants["Education_status"],
    normalize="index"
) * 100

In [ ]:
age_education = pd.crosstab(
    new_entrants["Age_Group"],
    new_entrants["Education_status"],
    normalize="index"
) * 100

age_education


In [ ]:
age_15_19 = new_entrants[
    (new_entrants["Q14AGE"] >= 15) &
    (new_entrants["Q14AGE"] < 20)
]

age_15_19["Education_status"].value_counts(normalize=True).sort_index() * 100

In [ ]:
age_15_19 = new_entrants[
    (new_entrants["Q14AGE"] >= 15) &
    (new_entrants["Q14AGE"] < 20)
]

age_15_19["Education_status"].value_counts(normalize=True).sort_index() * 100


In [ ]:
age_25_29 = new_entrants[
    (new_entrants["Q14AGE"] >= 25) &
    (new_entrants["Q14AGE"] < 30)
]

age_25_29["Education_status"].value_counts(normalize=True).sort_index() * 100

In [ ]:
age_education

In [ ]:
age_education[[4, 5]]

In [ ]:
status_5_change = age_education.loc["50-64", 5] - age_education.loc["15-19", 5]
status_5_change

In [ ]:
status_4_change = age_education.loc["50-64", 4] - age_education.loc["15-19", 4]
status_4_change

In [ ]:
new_entrants["Q13GENDER"].value_counts()

In [ ]:
pd.crosstab(
    new_entrants["Q13GENDER"],
    columns="count",
    normalize=True
) * 100

In [ ]:
gender_new_entrant = (
    df.groupby("Q13GENDER")["Unempl_Status"]
    .apply(lambda x: (x == "3").mean() * 100)
)

gender_new_entrant

In [ ]:
gender_education_new = (
    pd.crosstab(
        df["Education_status"],
        df["Q13GENDER"],
        values=(df["Unempl_Status"] == "3"),
        aggfunc="mean"
    ) * 100
)

gender_education_new

In [ ]:
gender_5 = gender_education_new.loc[5]

gender_5[2] - gender_5[1]

In [ ]:
gender_education_new["Female_minus_Male"] = (
    gender_education_new[2] - gender_education_new[1]
)

gender_education_new["Female_minus_Male"]

In [ ]:
gender_education_new["Female_minus_Male"].max()

In [ ]:
gender_education_new["Female_minus_Male"].min()

In [ ]:
age_gender_new = (
    pd.crosstab(
        new_entrants["Age_Group"],
        new_entrants["Q13GENDER"],
        normalize="index"
    ) * 100
)

age_gender_new

In [ ]:
age_gender_new["Female_minus_Male"] = (
    age_gender_new[2] - age_gender_new[1]
)

age_gender_new["Female_minus_Male"]

In [ ]:
age_gender_new["Female_minus_Male"].max()

In [ ]:
age_gender_new["Female_minus_Male"].min()

In [ ]:
age_gender_new["Female_minus_Male"].sort_values()

In [ ]:
weighted_age_gender = (
    new_entrants.groupby(["Age_Group", "Q13GENDER"])["Weight"]
    .sum()
    .groupby(level=0)
    .apply(lambda x: x / x.sum() * 100)
)

weighted_age_gender

In [ ]:
weighted_age_gender.sort_values()

In [ ]:
weighted_age_gender_df = weighted_age_gender.unstack()

weighted_age_gender_df["Female_minus_Male"] = (
    weighted_age_gender_df[2] - weighted_age_gender_df[1]
)

weighted_age_gender_df["Female_minus_Male"]

In [ ]:
[column for column in df.columns if "province" in column.lower()
 or "prov" in column.lower()
 or "geo" in column.lower()]

In [ ]:
df["Province"].value_counts().sort_index()

In [ ]:
df["Province"].value_counts().sort_values()

In [ ]:
df.groupby("Province")["Q13GENDER"].count().sort_index()

In [ ]:
province_new_entrant = (
    df.groupby("Province")["Unempl_Status"]
      .apply(lambda x: (x == "3").mean() * 100)
)

province_new_entrant

In [ ]:
# Weighted new-entrant share by province
# Denominator: respondents with a valid unemployment status in each province.

valid_unemployment = df[df["Unempl_Status"].notna()].copy()

weighted_province_new = (
    valid_unemployment.groupby("Province")
    .apply(
        lambda x: (
            x["Weight"] * (x["Unempl_Status"] == "3")
        ).sum()
        / x["Weight"].sum() * 100
    )
)

weighted_province_new


In [ ]:
province_comparison = pd.DataFrame({
    "Unweighted": province_new_entrant,
    "Weighted": weighted_province_new
})

province_comparison["Change"] = (
    province_comparison["Weighted"]
    - province_comparison["Unweighted"]
)

province_comparison.sort_values("Change", ascending=False)

In [ ]:
province_comparison["Change"].plot(kind="bar")

plt.title("Change in New-Entrant Rate After Weighting")
plt.xlabel("Province")
plt.ylabel("Weighted − Unweighted (percentage points)")
plt.axhline(0)
plt.show()

In [ ]:
province_names = {
    1: "Western Cape",
    2: "Eastern Cape",
    3: "Northern Cape",
    4: "Free State",
    5: "KwaZulu-Natal",
    6: "North West",
    7: "Gauteng",
    8: "Mpumalanga",
    9: "Limpopo"
}

province_comparison.index = province_comparison.index.map(province_names)

province_comparison

In [ ]:
province_comparison[["Unweighted", "Weighted"]].plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("New-Entrant Share by Province: Unweighted vs Weighted")
plt.xlabel("Province")
plt.ylabel("Percentage")
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
province_comparison.sort_values(
    "Weighted",
    ascending=False
)

In [ ]:
# Weighted new-entrant share by province and gender

province_gender_new = (
    valid_unemployment
    .groupby(["Province", "Q13GENDER"])
    .apply(
        lambda x: (
            x["Weight"] * (x["Unempl_Status"] == "3")
        ).sum()
        / x["Weight"].sum() * 100
    )
    .unstack()
)

province_gender_new


In [ ]:
province_gender_new["Female_minus_Male"] = (
    province_gender_new[2] - province_gender_new[1]
)

province_gender_new["Female_minus_Male"].sort_values(ascending=False)

In [ ]:
province_gender_new.index = province_gender_new.index.map(province_names)

In [ ]:
province_gender_new.loc[
    ["Mpumalanga", "Northern Cape", "North West"],
    [1, 2, "Female_minus_Male"]
]

In [ ]:
df["Underempl"].value_counts(dropna=False).sort_index()

In [ ]:
df["Underempl"] = df["Underempl"].replace(
    "1.79769313486232e+308",
    np.nan
)

In [ ]:
df["Underempl"].value_counts(dropna=False).sort_index()

In [ ]:
valid_underempl = df[df["Underempl"].notna()]

underempl_rate = (
    (valid_underempl["Underempl"] == "1").mean() * 100
)

underempl_rate

In [ ]:
education_underempl = (
    valid_underempl.groupby("Education_status")["Underempl"]
    .apply(lambda x: (x == "1").mean() * 100)
)

education_underempl

In [ ]:
education_underempl_count = (
    valid_underempl["Education_status"]
    .value_counts()
    .sort_index()
)

education_underempl_count

In [ ]:
gender_underempl = (
    valid_underempl.groupby("Q13GENDER")["Underempl"]
    .apply(lambda x: (x == "1").mean() * 100)
)

gender_underempl

In [ ]:
female_minus_male = gender_underempl[2] - gender_underempl[1]
female_minus_male

In [ ]:
province_underempl = (
    valid_underempl.groupby("Province")["Underempl"]
    .apply(lambda x: (x == "1").mean() * 100)
)

province_underempl

In [ ]:
province_underempl.sort_values()

In [ ]:
education_gender_underempl = (
    pd.crosstab(
        valid_underempl["Education_status"],
        valid_underempl["Q13GENDER"],
        values=(valid_underempl["Underempl"] == "1"),
        aggfunc="mean"
    ) * 100
)

education_gender_underempl

In [ ]:
education_gender_underempl["Female_minus_Male"] = (
    education_gender_underempl[2] - education_gender_underempl[1]
)

In [ ]:
education_gender_underempl["Female_minus_Male"]

In [ ]:
education_gender_underempl.loc[2, [1, 2, "Female_minus_Male"]]

In [ ]:
education_underempl.plot(kind="bar")

plt.title("Underemployment Rate by Education Status")
plt.xlabel("Education Status")
plt.ylabel("Underemployment Rate (%)")
plt.show()

In [ ]:
highest_rate = education_underempl.max()
lowest_rate = education_underempl.min()

highest_rate - lowest_rate

In [ ]:
vbins = [15, 20, 25, 30, 35, 40, 50, 65, 116]
labels = ["15-19", "20-24", "25-29", "30-34",
          "35-39", "40-49", "50-64", "65+"]

valid_underempl["Age_Group"] = pd.cut(
    valid_underempl["Q14AGE"],
    bins=bins,
    labels=labels,
    right=False
)

In [ ]:
age_underempl = (
    pd.crosstab(
        valid_underempl["Age_Group"],
        valid_underempl["Underempl"],
        normalize="index"
    ) * 100
)

age_underempl

In [ ]:
age_underempl["1"].plot(kind="bar")

plt.title("Underemployment Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Underemployment Rate (%)")
plt.show()

In [ ]:
age_gender_underempl = (
    pd.crosstab(
        valid_underempl["Age_Group"],
        valid_underempl["Q13GENDER"],
        values=(valid_underempl["Underempl"] == "1"),
        aggfunc="mean"
    ) * 100
)

age_gender_underempl

In [ ]:
age_gender_underempl["Female_minus_Male"] = (
    age_gender_underempl[2] - age_gender_underempl[1]
)

age_gender_underempl["Female_minus_Male"].sort_values(ascending=False)

In [ ]:
age_gender_underempl.loc[
    ["30-34", "20-24"],
    [1, 2, "Female_minus_Male"]
]

In [ ]:
province_age_underempl = (
    pd.crosstab(
        valid_underempl["Province"],
        valid_underempl["Age_Group"],
        values=(valid_underempl["Underempl"] == "1"),
        aggfunc="mean"
    ) * 100
)

province_age_underempl

In [ ]:
province_age_underempl["25-29"].sort_values()

In [ ]:
difference = (
    province_age_underempl.loc[4, "25-29"]
    - province_age_underempl.loc[1, "25-29"]
)

difference

In [ ]:
province_underempl.sort_values(ascending=False)

In [ ]:
province_gender_underempl = (
    pd.crosstab(
        valid_underempl["Province"],
        valid_underempl["Q13GENDER"],
        values=(valid_underempl["Underempl"] == "1"),
        aggfunc="mean"
    ) * 100
)

province_gender_underempl

In [ ]:
province_gender_underempl["Female_minus_Male"] = (
    province_gender_underempl[2] - province_gender_underempl[1]
)

province_gender_underempl["Female_minus_Male"].sort_values(ascending=False)

In [ ]:
province_gender_underempl.loc[
    [8, 9],
    [1, 2, "Female_minus_Male"]
]

In [ ]:
valid_unemployment["Unempl_Status"].value_counts(normalize=True) * 100

In [ ]:
unemployment_names = {
    "1": "Job loser",
    "2": "Job leaver",
    "3": "New entrant",
    "4": "Re-entrant",
    "5": "Other"
}

In [ ]:
valid_unemployment["Unempl_Status_Name"] = (
    valid_unemployment["Unempl_Status"].map(unemployment_names)
)

In [ ]:
valid_unemployment["Unempl_Status_Name"].value_counts()

In [ ]:
valid_unemployment["Unempl_Status_Name"].value_counts(normalize=True) * 100

In [ ]:
education_unemployment = (
    pd.crosstab(
        valid_unemployment["Education_status"],
        valid_unemployment["Unempl_Status_Name"],
        normalize="index"
    ) * 100
)

education_unemployment

In [ ]:
education_unemployment["New entrant"].sort_values(ascending=False)

In [ ]:
highest = education_unemployment["New entrant"].max()
lowest = education_unemployment["New entrant"].min()

difference = highest - lowest

difference

In [ ]:
gender_unemployment = (
    pd.crosstab(
        valid_unemployment["Q13GENDER"],
        valid_unemployment["Unempl_Status_Name"],
        normalize="index"
    ) * 100
)

gender_unemployment

In [ ]:
gender_unemployment["New entrant"]

In [ ]:
province_unemployment = (
    pd.crosstab(
        valid_unemployment["Province"],
        valid_unemployment["Unempl_Status_Name"],
        normalize="index"
    ) * 100
)

province_unemployment

In [ ]:
province_unemployment["New entrant"].sort_values(ascending=False)

In [ ]:
province_education_new = (
    pd.crosstab(
        valid_unemployment["Province"],
        valid_unemployment["Education_status"],
        values=(valid_unemployment["Unempl_Status_Name"] == "New entrant"),
        aggfunc="mean"
    ) * 100
)

province_education_new

In [ ]:
province_education_new.loc[9].sort_values(ascending=False)

In [ ]:
province_education_new.loc[:, [4, 5, 6]].sort_values(
    by=5,
    ascending=False
)

In [ ]:
province_education_new[5].max() - province_education_new[5].min()

In [ ]:
province_education_new[5].sort_values(ascending=False)

In [ ]:
province_education_new[6].sort_values(ascending=False)

In [ ]:
province_education_new.loc[9, [4, 5, 6]]

In [ ]:
province_education_new[5].sort_values(ascending=False)

In [ ]:
province_education_new.idxmax(axis=1)

In [ ]:
province_education_new[province_education_new.idxmax(axis=1) == 7]

In [ ]:
province_education_new[7].max() - province_education_new[7].min()

In [ ]:
province_education_new[7].sort_values(ascending=False)

In [ ]:
province_education_counts = pd.crosstab(
    valid_unemployment["Province"],
    valid_unemployment["Education_status"]
)

province_education_counts[7].sort_values(ascending=False)

In [ ]:
province_education_new[5].sort_values(ascending=False)

In [ ]:
province_gender_new.sort_values(
    "Female_minus_Male",
    ascending=False
)

In [ ]:
gender_education_new[
    [1, 2, "Female_minus_Male"]
].sort_values("Female_minus_Male", ascending=False)

In [ ]:
gender_education_new.loc[3]

In [ ]:
gender_education_new["Female_minus_Male"].agg(["min", "max"])

In [ ]:
weighted_province_new.sort_values(ascending=False)

In [ ]:
weighted_province_new.sort_values()

In [ ]:
weighted_province_new.plot(kind="bar")

plt.title("Weighted New-Entrant Rate by Province")
plt.xlabel("Province")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=0)
plt.show()

In [ ]:
province_names = {
    1: "Western Cape",
    2: "Eastern Cape",
    3: "Northern Cape",
    4: "Free State",
    5: "KwaZulu-Natal",
    6: "North West",
    7: "Gauteng",
    8: "Mpumalanga",
    9: "Limpopo"
}

weighted_province_named = weighted_province_new.rename(index=province_names)

In [ ]:
weighted_province_named = weighted_province_named.sort_values(
    ascending=False
)

In [ ]:
weighted_province_named.plot(kind="bar")

plt.title("Weighted New-Entrant Rate by Province")
plt.xlabel("Province")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
gap = weighted_province_named.max() - weighted_province_named.min()
gap

In [ ]:
weighted_province_named.head(3)

In [ ]:
gender_new_entrant


In [ ]:
gender_new_entrant.plot(kind="bar")
plt.title("New-Entrant Rate by Gender")
plt.xlabel("Gender")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks([0, 1], ["Male", "Female"], rotation=0)
plt.show()

In [ ]:
province_gender_new.sort_values(
    "Female_minus_Male",
    ascending=False
).head(3)

In [ ]:
province_gender_new.sort_values(
    "Female_minus_Male",
    ascending=True
).head(3)

In [ ]:
gap = province_gender_new["Female_minus_Male"].max() - province_gender_new["Female_minus_Male"].min()
gap

In [ ]:
gender_education_new[
    [1, 2]
].plot(kind="bar")

plt.title("New-Entrant Rate by Education Status and Gender")
plt.xlabel("Education Status")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=0)
plt.legend(["Male", "Female"])
plt.show()

In [ ]:
gender_education_new["Female_minus_Male"].max()

In [ ]:
gender_education_new["Female_minus_Male"].min()

In [ ]:
weighted_province_named.plot(kind="bar")
bars =  weighted_province_named.plot(kind="bar").containers[0]
for bar in bars:
    height = bar.get_height().round(2)
    plt.text(bar.get_x() + bar.get_width() / 2,height + 0.23, str(height))
plt.title("Weighted New-Entrant Rate by Province")
plt.xlabel("Province")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
ax = weighted_province_named.plot(kind="bar")

bars = ax.containers[0]

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.23,
        f"{height:.2f}",
        ha="center"
    )

plt.title("Weighted New-Entrant Rate by Province")
plt.xlabel("Province")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
province_gender_new.plot(kind="bar")
plt.title("New-Entrant Share by Province and Gender")
plt.xlabel("Province")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
gender_chart = province_gender_new[[1, 2]].rename(
    columns={1: "Male", 2: "Female"}
)

ax = gender_chart.plot(kind="bar")

plt.title("Weighted New-Entrant Rate by Gender and Province")
plt.xlabel("Province")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Gender")
plt.tight_layout()
plt.show()

In [ ]:
ax = gender_chart.plot(kind="bar")

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=4)

plt.title("Weighted New-Entrant Rate by Gender and Province")
plt.xlabel("Province")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Gender")
plt.tight_layout()
plt.show()

In [ ]:
weighted_new_entrant.plot(kind="bar")

plt.title("Weighted New-Entrant Rate by Education Status")
plt.xlabel("Education Status")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
education_names = {
    1: "No schooling",
    2: "Less than primary",
    3: "Primary completed",
    4: "Secondary not completed",
    5: "Secondary completed",
    6: "Tertiary",
    7: "Other"
}

In [ ]:
weighted_education_named = weighted_new_entrant.rename(
    index=education_names
)

In [ ]:
ax = weighted_education_named.plot(kind="bar")

plt.title("Weighted New-Entrant Rate by Education Status")
plt.xlabel("Education Status")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
ax = weighted_education_named.plot(kind="bar")

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.title("Weighted New-Entrant Rate by Education Status")
plt.xlabel("Education Status")
plt.ylabel("New-Entrant Rate (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Final validated results

The cells below contain the final weighted calculations used for the portfolio report. They use `valid_unemployment` so that the denominator contains only respondents with a recorded unemployment status.

The three main portfolio findings are:

- **Province:** Limpopo and Western Cape show different weighted new-entrant shares.
- **Gender by province:** the male/female difference varies across provinces.
- **Education:** secondary completed and primary completed show different weighted new-entrant shares.

These are descriptive differences in the QLFS 2026 Q1 sample and should not be interpreted as causal effects.


In [ ]:
education_check = (
    valid_unemployment.groupby("Education_status")
    .apply(
        lambda x: (x["Weight"] * (x["Unempl_Status"] == "3")).sum()
        / x["Weight"].sum() * 100
    )
)

education_check = education_check.rename(
    index=education_names
).sort_values(ascending=False)

print(education_check)

In [ ]:
gender_check = (
    valid_unemployment.groupby("Q13GENDER")
    .apply(
        lambda x: (x["Weight"] * (x["Unempl_Status"] == "3")).sum()
        / x["Weight"].sum() * 100
    )
)

gender_check = gender_check.rename(
    index={1: "Male", 2: "Female"}
)

print(gender_check)

In [ ]:
province_gender_check = (
    valid_unemployment
    .groupby(["Province", "Q13GENDER"])
    .apply(
        lambda x: (
            x["Weight"] * (x["Unempl_Status"] == "3")
        ).sum()
        / x["Weight"].sum() * 100
    )
)

province_gender_check = province_gender_check.unstack()
province_gender_check = province_gender_check.rename(
    columns={1: "Male", 2: "Female"}
)

province_gender_check = province_gender_check.rename(
    index=province_names
)

province_gender_check["Female_minus_Male"] = (
    province_gender_check["Female"]
    - province_gender_check["Male"]
)

print(province_gender_check.sort_values("Female_minus_Male",ascending=False))

In [ ]:
province_check = (
    valid_unemployment
    .groupby("Province")
    .apply(
        lambda x: (
            x["Weight"] * (x["Unempl_Status"] == "3")
        ).sum()
        / x["Weight"].sum() * 100
    )
)

province_check = province_check.rename(
    index=province_names
).sort_values(ascending=False)

print(province_check)

In [ ]:
ax = province_check.plot(kind="bar")

for bar in ax.containers[0]:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.5,
        f"{height:.2f}",
        ha="center"
    )

plt.title("Weighted New-Entrant Share by Province")
plt.xlabel("Province")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
province_gender_check

In [ ]:
gender_chart = province_gender_check[["Male", "Female"]]

ax = gender_chart.plot(kind="bar")

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.title("Weighted New-Entrant Share by Province and Gender")
plt.xlabel("Province")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=45)
plt.legend(title="Gender")
plt.tight_layout()
plt.show()

In [ ]:
education_check = (
    valid_unemployment
    .groupby("Education_status")
    .apply(
        lambda x: (
            x["Weight"] * (x["Unempl_Status"] == "3")
        ).sum()
        / x["Weight"].sum() * 100
    )
)

education_check = education_check.rename(index=education_names)
education_check = education_check.sort_values(ascending=False)

print(education_check)